# SplatAgent Pipeline — Colab GPU Runner

Generate 3D Gaussian Splats from images or video.

**Pipeline:** Input (images/video) → Frames → COLMAP (SfM) → gsplat Training → `.ply` Export

**Requirements:** GPU runtime (T4 or better). Go to **Runtime → Change runtime type → T4 GPU**.

## 0. Verify GPU & Install System Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi
print("\n" + "="*60)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install COLMAP and ffmpeg
!apt-get update -qq && apt-get install -y -qq colmap ffmpeg > /dev/null 2>&1
!colmap -h 2>&1 | head -2
!ffmpeg -version 2>&1 | head -1

In [ ]:
# Install Python dependencies
!pip install -q gsplat opencv-python-headless Pillow rich click scipy torchmetrics

import gsplat
print(f"gsplat: {getattr(gsplat, '__version__', 'installed')}")

## 1. Pipeline Code

The cells below write the full pipeline to `/content/pipeline/`. Run them all.

In [ ]:
import os
os.makedirs('/content/pipeline/splatagent_pipeline/stages', exist_ok=True)
os.makedirs('/content/pipeline/splatagent_pipeline/utils', exist_ok=True)
print('Directory structure created')

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/__init__.py
"""SplatAgent Pipeline — Generate 3D Gaussian Splats from images or video."""
__version__ = "0.1.0"

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/logging_setup.py
"""Structured logging with Rich console + file output."""
from __future__ import annotations
import logging
from pathlib import Path
from rich.console import Console
from rich.logging import RichHandler

_console = Console(stderr=True)

def setup_logging(run_dir: Path | None = None, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger("splatagent_pipeline")
    logger.setLevel(level)
    if logger.handlers:
        return logger
    console_handler = RichHandler(console=_console, show_time=True, show_path=False, markup=True, rich_tracebacks=True)
    console_handler.setLevel(level)
    logger.addHandler(console_handler)
    if run_dir is not None:
        run_dir.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(run_dir / "pipeline.log")
        file_handler.setLevel(level)
        file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"))
        logger.addHandler(file_handler)
    return logger

def get_logger() -> logging.Logger:
    return logging.getLogger("splatagent_pipeline")

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/config.py
"""Pipeline configuration — frozen dataclasses loaded from TOML."""
from __future__ import annotations
import dataclasses
import sys
from pathlib import Path
from typing import Literal

if sys.version_info >= (3, 11):
    import tomllib
else:
    try:
        import tomli as tomllib
    except ImportError:
        tomllib = None

@dataclasses.dataclass(frozen=True)
class VideoIngestConfig:
    fps: float = 2.0
    max_frames: int = 300
    blur_threshold: float = 100.0
    dedup_threshold: float = 0.95

@dataclasses.dataclass(frozen=True)
class ImageIngestConfig:
    max_dimension: int = 1600
    formats: tuple[str, ...] = ("jpg", "jpeg", "png", "webp", "tiff", "bmp")

@dataclasses.dataclass(frozen=True)
class IngestConfig:
    source_type: Literal["images", "video"] = "images"
    source_path: str = ""
    video: VideoIngestConfig = dataclasses.field(default_factory=VideoIngestConfig)
    images: ImageIngestConfig = dataclasses.field(default_factory=ImageIngestConfig)

@dataclasses.dataclass(frozen=True)
class SfMConfig:
    colmap_binary: str = "colmap"
    camera_model: str = "OPENCV"
    single_camera: bool = True
    matcher: Literal["exhaustive", "sequential", "vocab_tree"] = "exhaustive"
    vocab_tree_path: str = ""
    mapper_max_num_models: int = 1
    min_registered_ratio: float = 0.5

@dataclasses.dataclass(frozen=True)
class TrainConfig:
    iterations: int = 30_000
    sh_degree: int = 3
    position_lr_init: float = 0.00016
    position_lr_final: float = 0.0000016
    position_lr_max_steps: int = 30_000
    feature_lr: float = 0.0025
    opacity_lr: float = 0.05
    scaling_lr: float = 0.005
    rotation_lr: float = 0.001
    densify_from_iter: int = 500
    densify_until_iter: int = 15_000
    densify_interval: int = 100
    densify_grad_threshold: float = 0.0002
    prune_opacity_threshold: float = 0.005
    prune_scale_threshold: float = 10.0
    loss_lambda_dssim: float = 0.2
    checkpoint_interval: int = 5_000
    log_interval: int = 100
    eval_interval: int = 1_000
    white_background: bool = False

@dataclasses.dataclass(frozen=True)
class ExportConfig:
    ply_format: str = "standard"
    include_sh_coefficients: bool = True
    max_sh_degree: int = 3

@dataclasses.dataclass(frozen=True)
class RunConfig:
    scene_id: str = ""
    output_dir: str = "./runs"
    resume: bool = True
    seed: int = 42

@dataclasses.dataclass(frozen=True)
class PipelineConfig:
    run: RunConfig = dataclasses.field(default_factory=RunConfig)
    ingest: IngestConfig = dataclasses.field(default_factory=IngestConfig)
    sfm: SfMConfig = dataclasses.field(default_factory=SfMConfig)
    train: TrainConfig = dataclasses.field(default_factory=TrainConfig)
    export: ExportConfig = dataclasses.field(default_factory=ExportConfig)

def config_to_dict(config: PipelineConfig) -> dict:
    return dataclasses.asdict(config)

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/manifest.py
"""Manifest schema and JSON serialization."""
from __future__ import annotations
import json
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

@dataclass
class SourceInfo:
    type: str = ""
    frame_count: int = 0
    image_dimensions: tuple[int, int] = (0, 0)

@dataclass
class SfMInfo:
    registered_images: int = 0
    sparse_points: int = 0
    mean_reprojection_error: float = 0.0

@dataclass
class TrainingInfo:
    iterations: int = 0
    final_gaussians: int = 0
    final_psnr: float = 0.0
    final_ssim: float = 0.0
    training_time_seconds: float = 0.0

@dataclass
class BoundsInfo:
    min: list[float] = field(default_factory=lambda: [0.0, 0.0, 0.0])
    max: list[float] = field(default_factory=lambda: [0.0, 0.0, 0.0])

@dataclass
class OutputInfo:
    ply_file: str = "scene.ply"
    ply_size_bytes: int = 0
    coordinate_convention: str = "colmap_y_down"

@dataclass
class ToolchainInfo:
    python_version: str = ""
    gsplat_version: str = ""
    colmap_version: str = ""
    torch_version: str = ""
    cuda_version: str = ""
    pipeline_version: str = ""
    ffmpeg_version: str = ""

@dataclass
class Manifest:
    schema_version: str = "1.0"
    scene_id: str = ""
    created_at: str = ""
    source: SourceInfo = field(default_factory=SourceInfo)
    sfm: SfMInfo = field(default_factory=SfMInfo)
    training: TrainingInfo = field(default_factory=TrainingInfo)
    bounds: BoundsInfo = field(default_factory=BoundsInfo)
    output: OutputInfo = field(default_factory=OutputInfo)
    toolchain: ToolchainInfo = field(default_factory=ToolchainInfo)
    config_snapshot: dict = field(default_factory=dict)

    def write(self, path: Path) -> None:
        if not self.created_at:
            self.created_at = datetime.now(timezone.utc).isoformat()
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2, default=str)

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/utils/__init__.py


In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/stages/__init__.py


In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/utils/ply_io.py
"""PLY read/write for standard 3D Gaussian Splat format."""
from __future__ import annotations
import struct
from pathlib import Path
import numpy as np

def write_ply(
    path: Path,
    means: np.ndarray,
    scales: np.ndarray,
    rotations: np.ndarray,
    opacities: np.ndarray,
    sh_dc: np.ndarray,
    sh_rest: np.ndarray | None = None,
    sh_degree: int = 3,
) -> int:
    n = means.shape[0]
    if opacities.ndim == 1:
        opacities = opacities[:, None]
    num_sh_rest = 3 * ((sh_degree + 1) ** 2 - 1)
    if sh_rest is not None:
        sh_rest_flat = sh_rest.reshape(n, -1).astype(np.float32)
    else:
        sh_rest_flat = np.zeros((n, num_sh_rest), dtype=np.float32)
    normals = np.zeros((n, 3), dtype=np.float32)
    header_lines = [
        "ply", "format binary_little_endian 1.0", f"element vertex {n}",
        "property float x", "property float y", "property float z",
        "property float nx", "property float ny", "property float nz",
        "property float f_dc_0", "property float f_dc_1", "property float f_dc_2",
    ]
    for i in range(num_sh_rest):
        header_lines.append(f"property float f_rest_{i}")
    header_lines += [
        "property float opacity",
        "property float scale_0", "property float scale_1", "property float scale_2",
        "property float rot_0", "property float rot_1", "property float rot_2", "property float rot_3",
        "end_header",
    ]
    header = "\n".join(header_lines) + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        for i in range(n):
            f.write(struct.pack("<3f", *means[i]))
            f.write(struct.pack("<3f", *normals[i]))
            f.write(struct.pack("<3f", *sh_dc[i]))
            f.write(struct.pack(f"<{num_sh_rest}f", *sh_rest_flat[i]))
            f.write(struct.pack("<f", opacities[i, 0]))
            f.write(struct.pack("<3f", *scales[i]))
            f.write(struct.pack("<4f", *rotations[i]))
    return n

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/utils/colmap_io.py
"""Read COLMAP binary model files."""
from __future__ import annotations
import struct
from pathlib import Path
import numpy as np

CAMERA_MODEL_IDS = {
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4),
    3: ("RADIAL", 5), 4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8),
    6: ("FULL_OPENCV", 12), 7: ("FOV", 5), 8: ("SIMPLE_RADIAL_FISHEYE", 4),
    9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12),
}

def read_cameras_binary(path: Path) -> dict[int, dict]:
    cameras = {}
    with open(path, "rb") as f:
        num_cameras = struct.unpack("<Q", f.read(8))[0]
        for _ in range(num_cameras):
            camera_id = struct.unpack("<I", f.read(4))[0]
            model_id = struct.unpack("<i", f.read(4))[0]
            width = struct.unpack("<Q", f.read(8))[0]
            height = struct.unpack("<Q", f.read(8))[0]
            model_name, num_params = CAMERA_MODEL_IDS[model_id]
            params = struct.unpack(f"<{num_params}d", f.read(8 * num_params))
            cameras[camera_id] = {"model": model_name, "model_id": model_id, "width": width, "height": height, "params": list(params)}
    return cameras

def read_images_binary(path: Path) -> dict[int, dict]:
    images = {}
    with open(path, "rb") as f:
        num_images = struct.unpack("<Q", f.read(8))[0]
        for _ in range(num_images):
            image_id = struct.unpack("<I", f.read(4))[0]
            qvec = struct.unpack("<4d", f.read(32))
            tvec = struct.unpack("<3d", f.read(24))
            camera_id = struct.unpack("<I", f.read(4))[0]
            name_chars = []
            while True:
                ch = f.read(1)
                if ch == b"\x00": break
                name_chars.append(ch.decode("ascii"))
            name = "".join(name_chars)
            num_points2D = struct.unpack("<Q", f.read(8))[0]
            xys, point3D_ids = [], []
            for _ in range(num_points2D):
                x, y = struct.unpack("<2d", f.read(16))
                p3d_id = struct.unpack("<q", f.read(8))[0]
                xys.append((x, y))
                point3D_ids.append(p3d_id)
            images[image_id] = {"qvec": np.array(qvec), "tvec": np.array(tvec), "camera_id": camera_id, "name": name, "xys": np.array(xys) if xys else np.empty((0, 2)), "point3D_ids": np.array(point3D_ids, dtype=np.int64)}
    return images

def read_points3D_binary(path: Path) -> dict[int, dict]:
    points3D = {}
    with open(path, "rb") as f:
        num_points = struct.unpack("<Q", f.read(8))[0]
        for _ in range(num_points):
            point3D_id = struct.unpack("<Q", f.read(8))[0]
            xyz = struct.unpack("<3d", f.read(24))
            rgb = struct.unpack("<3B", f.read(3))
            error = struct.unpack("<d", f.read(8))[0]
            track_length = struct.unpack("<Q", f.read(8))[0]
            track = []
            for _ in range(track_length):
                img_id = struct.unpack("<I", f.read(4))[0]
                p2d_idx = struct.unpack("<I", f.read(4))[0]
                track.append((img_id, p2d_idx))
            points3D[point3D_id] = {"xyz": np.array(xyz), "rgb": np.array(rgb, dtype=np.uint8), "error": error, "track": track}
    return points3D

def read_colmap_model(model_dir: Path) -> tuple[dict, dict, dict]:
    for name in ["cameras.bin", "images.bin", "points3D.bin"]:
        if not (model_dir / name).exists():
            raise FileNotFoundError(f"COLMAP model file not found: {model_dir / name}")
    return (read_cameras_binary(model_dir / "cameras.bin"),
            read_images_binary(model_dir / "images.bin"),
            read_points3D_binary(model_dir / "points3D.bin"))

def qvec_to_rotmat(qvec: np.ndarray) -> np.ndarray:
    w, x, y, z = qvec
    return np.array([
        [1-2*y*y-2*z*z, 2*x*y-2*w*z, 2*x*z+2*w*y],
        [2*x*y+2*w*z, 1-2*x*x-2*z*z, 2*y*z-2*w*x],
        [2*x*z-2*w*y, 2*y*z+2*w*x, 1-2*x*x-2*y*y],
    ])

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/utils/image_utils.py
"""Image processing: blur detection, resize, dedup."""
from __future__ import annotations
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from ..logging_setup import get_logger

class ImageError(Exception): pass

def compute_blur_score(image_path: Path) -> float:
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None: raise ImageError(f"Cannot read image: {image_path}")
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

def resize_if_needed(image_path: Path, max_dimension: int, output_path: Path) -> tuple[int, int]:
    img = Image.open(image_path)
    w, h = img.size
    if max(w, h) > max_dimension:
        scale = max_dimension / max(w, h)
        w, h = int(w * scale), int(h * scale)
        img = img.resize((w, h), Image.LANCZOS)
    if img.mode != "RGB": img = img.convert("RGB")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(output_path, "JPEG", quality=95)
    return (w, h)

def compute_image_hash(image_path: Path, hash_size: int = 16) -> np.ndarray:
    img = Image.open(image_path).convert("L").resize((hash_size, hash_size), Image.LANCZOS)
    pixels = np.array(img)
    return (pixels > pixels.mean()).flatten()

def hash_similarity(h1: np.ndarray, h2: np.ndarray) -> float:
    return float(np.mean(h1 == h2))

def filter_frames(frame_paths: list[Path], blur_threshold: float = 100.0, dedup_threshold: float = 0.95) -> tuple[list[Path], list[dict]]:
    log = get_logger()
    kept, skipped = [], []
    prev_hash = None
    for frame_path in frame_paths:
        try:
            blur_score = compute_blur_score(frame_path)
        except ImageError:
            skipped.append({"path": str(frame_path), "reason": "unreadable"})
            continue
        if blur_score < blur_threshold:
            skipped.append({"path": str(frame_path), "reason": "blur", "score": round(blur_score, 2)})
            continue
        if dedup_threshold < 1.0:
            current_hash = compute_image_hash(frame_path)
            if prev_hash is not None and hash_similarity(prev_hash, current_hash) >= dedup_threshold:
                skipped.append({"path": str(frame_path), "reason": "duplicate"})
                continue
            prev_hash = current_hash
        kept.append(frame_path)
    log.info(f"Frame filtering: {len(kept)} kept, {len(skipped)} skipped")
    return kept, skipped

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/utils/video_utils.py
"""FFmpeg wrapper for video frame extraction."""
from __future__ import annotations
import json, shutil, subprocess
from pathlib import Path
from ..logging_setup import get_logger

class VideoError(Exception): pass

def check_ffmpeg() -> str:
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None: raise VideoError("ffmpeg not found on PATH")
    result = subprocess.run([ffmpeg, "-version"], capture_output=True, text=True)
    return result.stdout.split("\n")[0]

def get_video_info(video_path: Path) -> dict:
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None: raise VideoError("ffprobe not found")
    cmd = [ffprobe, "-v", "quiet", "-print_format", "json", "-show_streams", "-show_format", str(video_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0: raise VideoError(f"ffprobe failed: {result.stderr}")
    data = json.loads(result.stdout)
    video_stream = next((s for s in data.get("streams", []) if s.get("codec_type") == "video"), None)
    if not video_stream: raise VideoError(f"No video stream in {video_path}")
    duration = float(data.get("format", {}).get("duration", 0))
    num, den = map(int, video_stream.get("r_frame_rate", "0/1").split("/"))
    return {"duration": duration, "width": int(video_stream.get("width", 0)), "height": int(video_stream.get("height", 0)), "native_fps": num / den if den else 0, "codec": video_stream.get("codec_name", "unknown")}

def extract_frames(video_path: Path, output_dir: Path, fps: float = 2.0, max_frames: int = 300) -> list[Path]:
    log = get_logger()
    if not video_path.is_file(): raise VideoError(f"Video not found: {video_path}")
    output_dir.mkdir(parents=True, exist_ok=True)
    info = get_video_info(video_path)
    log.info(f"Video: {info['width']}x{info['height']}, {info['duration']:.1f}s, {info['native_fps']:.1f}fps")
    expected = int(info["duration"] * fps)
    if expected > max_frames:
        fps = max_frames / info["duration"]
        log.info(f"Adjusting to {fps:.2f} fps to cap at {max_frames} frames")
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-vf", f"fps={fps}", "-qscale:v", "2", str(output_dir / "frame_%06d.jpg")]
    log.info(f"Extracting: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0: raise VideoError(f"ffmpeg failed: {result.stderr[:500]}")
    frames = sorted(output_dir.glob("frame_*.jpg"))
    if not frames: raise VideoError("No output frames")
    if len(frames) > max_frames:
        for f in frames[max_frames:]: f.unlink()
        frames = frames[:max_frames]
    log.info(f"Extracted {len(frames)} frames")
    return frames

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/stages/ingest.py
"""Stage 1 — Ingestion."""
from __future__ import annotations
import json, shutil
from pathlib import Path
from ..config import PipelineConfig
from ..logging_setup import get_logger
from ..utils.image_utils import filter_frames, resize_if_needed
from ..utils.video_utils import VideoError, check_ffmpeg, extract_frames

class IngestError(Exception): pass
STAGE_MARKER = "ingest_report.json"
MIN_FRAMES = 3

def is_complete(run_dir: Path) -> bool:
    return (run_dir / STAGE_MARKER).exists() and (run_dir / "images").exists()

def run_ingest(config: PipelineConfig, run_dir: Path) -> dict:
    log = get_logger()
    images_dir = run_dir / "images"
    if config.run.resume and is_complete(run_dir):
        log.info("[bold green]Stage 1 (Ingest): skipping — already complete[/]")
        with open(run_dir / STAGE_MARKER) as f: return json.load(f)
    log.info("[bold cyan]Stage 1 (Ingest): starting[/]")
    images_dir.mkdir(parents=True, exist_ok=True)
    source_type = config.ingest.source_type
    source_path = Path(config.ingest.source_path)
    if source_type == "video":
        report = _ingest_video(config, source_path, images_dir)
    elif source_type == "images":
        report = _ingest_images(config, source_path, images_dir)
    else:
        raise IngestError(f"Unknown source_type: {source_type}")
    final_frames = sorted(images_dir.glob("frame_*.jpg"))
    if len(final_frames) < MIN_FRAMES:
        raise IngestError(f"Only {len(final_frames)} frames after filtering; need at least {MIN_FRAMES}.")
    report["final_frame_count"] = len(final_frames)
    report["source_type"] = source_type
    report["source_path"] = str(source_path)
    from PIL import Image
    with Image.open(final_frames[0]) as img:
        report["image_dimensions"] = list(img.size)
    with open(run_dir / STAGE_MARKER, "w") as f: json.dump(report, f, indent=2)
    log.info(f"[bold green]Stage 1 (Ingest): complete — {report['final_frame_count']} frames[/]")
    return report

def _ingest_video(config, video_path, images_dir):
    if not video_path.is_file(): raise IngestError(f"Video not found: {video_path}")
    try: ffmpeg_version = check_ffmpeg()
    except VideoError as e: raise IngestError(str(e)) from e
    vc = config.ingest.video
    raw_dir = images_dir.parent / "_raw_frames"
    raw_dir.mkdir(parents=True, exist_ok=True)
    try: raw_frames = extract_frames(video_path, raw_dir, fps=vc.fps, max_frames=vc.max_frames)
    except VideoError as e: raise IngestError(f"Frame extraction failed: {e}") from e
    kept, skipped = filter_frames(raw_frames, blur_threshold=vc.blur_threshold, dedup_threshold=vc.dedup_threshold)
    ic = config.ingest.images
    for i, fp in enumerate(kept, 1):
        resize_if_needed(fp, ic.max_dimension, images_dir / f"frame_{i:06d}.jpg")
    shutil.rmtree(raw_dir, ignore_errors=True)
    return {"raw_frame_count": len(raw_frames), "skipped_count": len(skipped), "ffmpeg_version": ffmpeg_version}

def _ingest_images(config, source_dir, images_dir):
    log = get_logger()
    if not source_dir.is_dir(): raise IngestError(f"Image dir not found: {source_dir}")
    ic = config.ingest.images
    valid_exts = set(f".{fmt}" for fmt in ic.formats)
    raw_paths = sorted(f for f in source_dir.iterdir() if f.suffix.lower() in valid_exts and f.is_file())
    if not raw_paths: raise IngestError(f"No valid images in {source_dir}")
    log.info(f"Found {len(raw_paths)} images")
    vc = config.ingest.video
    kept, skipped = filter_frames(raw_paths, blur_threshold=vc.blur_threshold, dedup_threshold=1.0)
    for i, fp in enumerate(kept, 1):
        resize_if_needed(fp, ic.max_dimension, images_dir / f"frame_{i:06d}.jpg")
    return {"raw_image_count": len(raw_paths), "skipped_count": len(skipped)}

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/stages/sfm.py
"""Stage 2 — Structure from Motion (COLMAP)."""
from __future__ import annotations
import json, shutil, subprocess
from pathlib import Path
from ..config import PipelineConfig
from ..logging_setup import get_logger
from ..utils.colmap_io import read_colmap_model

class SfMError(Exception): pass
STAGE_MARKER = "sfm_report.json"

def is_complete(run_dir: Path) -> bool:
    model_dir = run_dir / "sparse" / "0"
    return (run_dir / STAGE_MARKER).exists() and all((model_dir / n).exists() for n in ["cameras.bin", "images.bin", "points3D.bin"])

def _run_colmap_cmd(cmd, step_name):
    log = get_logger()
    log.info(f"  COLMAP {step_name}: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    if result.returncode != 0:
        raise SfMError(f"COLMAP {step_name} failed (exit {result.returncode}).\nstderr: {result.stderr[-1000:]}")

def run_sfm(config: PipelineConfig, run_dir: Path) -> dict:
    log = get_logger()
    if config.run.resume and is_complete(run_dir):
        log.info("[bold green]Stage 2 (SfM): skipping — already complete[/]")
        with open(run_dir / STAGE_MARKER) as f: return json.load(f)
    log.info("[bold cyan]Stage 2 (SfM): starting[/]")
    images_dir = run_dir / "images"
    if not images_dir.exists(): raise SfMError("No images found. Run ingest first.")
    sc = config.sfm
    colmap = shutil.which(sc.colmap_binary) or sc.colmap_binary
    db = run_dir / "database.db"
    sparse_dir = run_dir / "sparse"
    sparse_dir.mkdir(parents=True, exist_ok=True)
    _run_colmap_cmd([colmap, "feature_extractor", "--database_path", str(db), "--image_path", str(images_dir), "--ImageReader.camera_model", sc.camera_model, "--ImageReader.single_camera", "1" if sc.single_camera else "0", "--SiftExtraction.use_gpu", "1"], "feature_extractor")
    _run_colmap_cmd([colmap, f"{sc.matcher}_matcher", "--database_path", str(db), "--SiftMatching.use_gpu", "1"], f"{sc.matcher}_matcher")
    _run_colmap_cmd([colmap, "mapper", "--database_path", str(db), "--image_path", str(images_dir), "--output_path", str(sparse_dir), "--Mapper.max_num_models", str(sc.mapper_max_num_models)], "mapper")
    model_dir = sparse_dir / "0"
    if not model_dir.exists():
        raise SfMError("COLMAP mapper produced no reconstruction. Images may lack texture or overlap.")
    cameras, images, points3D = read_colmap_model(model_dir)
    total = len(list(images_dir.glob("frame_*.jpg")))
    registered = len(images)
    ratio = registered / total if total > 0 else 0.0
    if ratio < sc.min_registered_ratio:
        raise SfMError(f"Only {registered}/{total} images registered ({ratio:.1%}). Min: {sc.min_registered_ratio:.0%}")
    errors = [p["error"] for p in points3D.values()]
    report = {"total_images": total, "registered_images": registered, "sparse_points": len(points3D), "mean_reprojection_error": round(sum(errors)/len(errors), 4) if errors else 0}
    with open(run_dir / STAGE_MARKER, "w") as f: json.dump(report, f, indent=2)
    log.info(f"[bold green]Stage 2 (SfM): complete — {registered}/{total} images, {len(points3D)} points[/]")
    return report

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/stages/train.py
"""Stage 3 — Gaussian Splat training with gsplat."""
from __future__ import annotations
import json, math, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import numpy as np
import torch
import torch.nn.functional as F
from ..config import PipelineConfig
from ..logging_setup import get_logger
from ..utils.colmap_io import qvec_to_rotmat, read_colmap_model

class TrainError(Exception): pass
STAGE_MARKER = "train_report.json"

@dataclass
class GaussianModel:
    means: torch.Tensor
    scales: torch.Tensor
    rotations: torch.Tensor
    opacities: torch.Tensor
    sh_dc: torch.Tensor
    sh_rest: torch.Tensor
    grad_accum: torch.Tensor = field(default=None, repr=False)
    grad_count: torch.Tensor = field(default=None, repr=False)

    @property
    def num_gaussians(self): return self.means.shape[0]
    @property
    def sh_degree(self): return int(math.sqrt(self.sh_rest.shape[1] + 1)) - 1
    def all_sh(self): return torch.cat([self.sh_dc, self.sh_rest], dim=1)

    def param_groups(self, config):
        tc = config.train
        return [
            {"params": [self.means], "lr": tc.position_lr_init, "name": "means"},
            {"params": [self.sh_dc], "lr": tc.feature_lr, "name": "sh_dc"},
            {"params": [self.sh_rest], "lr": tc.feature_lr / 20.0, "name": "sh_rest"},
            {"params": [self.opacities], "lr": tc.opacity_lr, "name": "opacities"},
            {"params": [self.scales], "lr": tc.scaling_lr, "name": "scales"},
            {"params": [self.rotations], "lr": tc.rotation_lr, "name": "rotations"},
        ]

    def reset_densification_stats(self):
        n, device = self.num_gaussians, self.means.device
        self.grad_accum = torch.zeros(n, 1, device=device)
        self.grad_count = torch.zeros(n, 1, device=device, dtype=torch.int32)

def _inverse_sigmoid(x): return math.log(x / (1 - x))

def init_from_colmap(model_dir, sh_degree, device="cuda"):
    log = get_logger()
    cameras, images, points3D = read_colmap_model(model_dir)
    if len(points3D) < 10:
        raise TrainError(f"Only {len(points3D)} COLMAP points. Need >= 10.")
    positions = np.array([p["xyz"] for p in points3D.values()], dtype=np.float32)
    colors = np.array([p["rgb"] for p in points3D.values()], dtype=np.float32) / 255.0
    n = positions.shape[0]
    log.info(f"Initializing {n} Gaussians from COLMAP")
    from scipy.spatial import KDTree
    tree = KDTree(positions)
    dists, _ = tree.query(positions, k=4)
    avg_dist = np.mean(dists[:, 1:], axis=1)
    log_scales = np.log(np.clip(avg_dist, 1e-7, None))[:, None].repeat(3, axis=1).astype(np.float32)
    C0 = 0.28209479177387814
    sh_dc = ((colors - 0.5) / C0).astype(np.float32)
    k = (sh_degree + 1) ** 2 - 1
    model = GaussianModel(
        means=torch.tensor(positions, device=device).requires_grad_(True),
        scales=torch.tensor(log_scales, device=device).requires_grad_(True),
        rotations=torch.tensor(np.tile([1,0,0,0], (n,1)).astype(np.float32), device=device).requires_grad_(True),
        opacities=torch.full((n,1), _inverse_sigmoid(0.1), device=device).requires_grad_(True),
        sh_dc=torch.tensor(sh_dc[:, None, :], device=device).requires_grad_(True),
        sh_rest=torch.zeros(n, k, 3, device=device).requires_grad_(True),
    )
    model.reset_densification_stats()
    return model

@dataclass
class CameraInfo:
    width: int; height: int; K: np.ndarray; world_to_cam: np.ndarray; image_path: Path; image_name: str

def load_cameras(model_dir, images_dir):
    cameras_data, images_data, _ = read_colmap_model(model_dir)
    cam_infos = []
    for img in images_data.values():
        cam = cameras_data[img["camera_id"]]
        params, w, h = cam["params"], cam["width"], cam["height"]
        model_name = cam["model"]
        if model_name in ("SIMPLE_PINHOLE", "SIMPLE_RADIAL", "RADIAL"):
            fx = fy = params[0]; cx, cy = params[1], params[2]
        elif model_name == "PINHOLE":
            fx, fy, cx, cy = params[0], params[1], params[2], params[3]
        else:
            fx, fy = params[0], params[1] if len(params) > 1 else params[0]
            cx = params[2] if len(params) > 2 else w / 2.0
            cy = params[3] if len(params) > 3 else h / 2.0
        K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]], dtype=np.float64)
        R = qvec_to_rotmat(img["qvec"])
        w2c = np.eye(4, dtype=np.float64)
        w2c[:3,:3] = R; w2c[:3,3] = img["tvec"]
        cam_infos.append(CameraInfo(width=w, height=h, K=K, world_to_cam=w2c, image_path=images_dir/img["name"], image_name=img["name"]))
    return cam_infos

def load_gt_image(cam, device="cuda"):
    from PIL import Image
    img = np.array(Image.open(cam.image_path).convert("RGB"), dtype=np.float32) / 255.0
    return torch.tensor(img, device=device, dtype=torch.float32).unsqueeze(0)

def get_position_lr(step, config):
    tc = config.train
    if step >= tc.position_lr_max_steps: return tc.position_lr_final
    t = step / tc.position_lr_max_steps
    return tc.position_lr_init * (tc.position_lr_final / tc.position_lr_init) ** t

def _ssim(img1, img2, window_size=11):
    img1 = img1.permute(0,3,1,2); img2 = img2.permute(0,3,1,2)
    C = img1.shape[1]
    sigma = 1.5
    coords = torch.arange(window_size, dtype=torch.float32, device=img1.device) - window_size // 2
    g = torch.exp(-(coords**2) / (2*sigma**2)); g = g / g.sum()
    window = (g.unsqueeze(1) * g.unsqueeze(0)).unsqueeze(0).unsqueeze(0).expand(C,1,-1,-1)
    pad = window_size // 2
    mu1 = F.conv2d(img1, window, padding=pad, groups=C)
    mu2 = F.conv2d(img2, window, padding=pad, groups=C)
    sigma1_sq = F.conv2d(img1*img1, window, padding=pad, groups=C) - mu1**2
    sigma2_sq = F.conv2d(img2*img2, window, padding=pad, groups=C) - mu2**2
    sigma12 = F.conv2d(img1*img2, window, padding=pad, groups=C) - mu1*mu2
    C1, C2 = 0.01**2, 0.03**2
    return ((2*mu1*mu2+C1)*(2*sigma12+C2) / ((mu1**2+mu2**2+C1)*(sigma1_sq+sigma2_sq+C2))).mean()

def _densify_and_prune(model, optimizer, config, iteration):
    tc = config.train
    stats = {"split": 0, "clone": 0, "prune": 0}
    avg_grad = (model.grad_accum / model.grad_count.clamp(min=1)).squeeze(-1)
    big_grad = avg_grad >= tc.densify_grad_threshold
    scales_exp = torch.exp(model.scales)
    large_scale = scales_exp.max(dim=1).values > 0.01
    split_mask = big_grad & large_scale
    clone_mask = big_grad & ~large_scale
    new_tensors = {k: [getattr(model, k)] for k in ["means","scales","rotations","opacities","sh_dc","sh_rest"]}
    if clone_mask.any():
        stats["clone"] = clone_mask.sum().item()
        for k in new_tensors: new_tensors[k].append(getattr(model, k)[clone_mask].clone())
    if split_mask.any():
        stats["split"] = split_mask.sum().item()
        for k in ["means","scales","rotations","opacities","sh_dc","sh_rest"]:
            t = getattr(model, k)[split_mask]
            if k == "scales": t = t - math.log(1.6)
            new_tensors[k].append(t.repeat(2, *([1]*(t.ndim-1))))
    # Prune
    opacity_act = torch.sigmoid(model.opacities.squeeze(-1))
    prune = (opacity_act < tc.prune_opacity_threshold) | (scales_exp.max(1).values > tc.prune_scale_threshold)
    if split_mask.any(): prune = prune | split_mask
    stats["prune"] = prune.sum().item()
    keep = ~prune
    # Apply: keep + new
    final = {}
    for k in new_tensors:
        new_tensors[k][0] = new_tensors[k][0][keep]
        final[k] = torch.cat(new_tensors[k], dim=0)
    for k in final:
        setattr(model, k, final[k].detach().requires_grad_(True))
    optimizer.state.clear()
    for pg in optimizer.param_groups:
        pg["params"] = [getattr(model, pg["name"])]
    model.reset_densification_stats()
    return stats

def save_checkpoint(model, optimizer, iteration, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"iteration": iteration, "means": model.means.detach().cpu(), "scales": model.scales.detach().cpu(), "rotations": model.rotations.detach().cpu(), "opacities": model.opacities.detach().cpu(), "sh_dc": model.sh_dc.detach().cpu(), "sh_rest": model.sh_rest.detach().cpu(), "optimizer_state": optimizer.state_dict()}, path)

def load_checkpoint(path, config, device="cuda"):
    data = torch.load(path, map_location=device, weights_only=True)
    model = GaussianModel(
        means=data["means"].to(device).requires_grad_(True),
        scales=data["scales"].to(device).requires_grad_(True),
        rotations=data["rotations"].to(device).requires_grad_(True),
        opacities=data["opacities"].to(device).requires_grad_(True),
        sh_dc=data["sh_dc"].to(device).requires_grad_(True),
        sh_rest=data["sh_rest"].to(device).requires_grad_(True),
    )
    model.reset_densification_stats()
    return model, data.get("optimizer_state"), data["iteration"]

def is_complete(run_dir):
    return (run_dir / STAGE_MARKER).exists() and (run_dir / "train" / "final_model.pt").exists()

def run_train(config, run_dir):
    log = get_logger()
    if config.run.resume and is_complete(run_dir):
        log.info("[bold green]Stage 3 (Train): skipping — already complete[/]")
        with open(run_dir / STAGE_MARKER) as f: return json.load(f)
    log.info("[bold cyan]Stage 3 (Train): starting[/]")
    if not torch.cuda.is_available():
        raise TrainError("CUDA not available. GPU required.")
    device = "cuda"
    tc = config.train
    try:
        from gsplat import rasterization
    except ImportError:
        raise TrainError("gsplat not installed. pip install gsplat")
    model_dir = run_dir / "sparse" / "0"
    images_dir = run_dir / "images"
    cam_infos = load_cameras(model_dir, images_dir)
    log.info(f"Loaded {len(cam_infos)} training cameras")
    train_dir = run_dir / "train"
    ckpt_dir = train_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    start_iter, optimizer_state = 0, None
    existing_ckpts = sorted(ckpt_dir.glob("iter_*.pt"))
    if config.run.resume and existing_ckpts:
        log.info(f"Resuming from {existing_ckpts[-1].name}")
        model, optimizer_state, start_iter = load_checkpoint(existing_ckpts[-1], config, device)
        start_iter += 1
    else:
        model = init_from_colmap(model_dir, tc.sh_degree, device)
    optimizer = torch.optim.Adam(model.param_groups(config), eps=1e-15)
    if optimizer_state:
        try: optimizer.load_state_dict(optimizer_state)
        except: log.warning("Could not restore optimizer state")
    torch.manual_seed(config.run.seed); np.random.seed(config.run.seed)
    bg = torch.tensor([1.,1.,1.] if tc.white_background else [0.,0.,0.], device=device)
    metrics_log = []
    t_start = time.time()
    log.info(f"Training: {tc.iterations} iters, {model.num_gaussians} Gaussians, SH degree {tc.sh_degree}")
    for iteration in range(start_iter, tc.iterations):
        lr = get_position_lr(iteration, config)
        for pg in optimizer.param_groups:
            if pg["name"] == "means": pg["lr"] = lr
        cam = cam_infos[np.random.randint(len(cam_infos))]
        gt_image = load_gt_image(cam, device)
        viewmat = torch.tensor(cam.world_to_cam, device=device, dtype=torch.float32).unsqueeze(0)
        K = torch.tensor(cam.K, device=device, dtype=torch.float32).unsqueeze(0)
        sh_coeffs = model.all_sh()
        try:
            renders, alphas, meta = rasterization(
                means=model.means, quats=model.rotations, scales=torch.exp(model.scales),
                opacities=torch.sigmoid(model.opacities.squeeze(-1)),
                colors=sh_coeffs, viewmats=viewmat, Ks=K,
                width=cam.width, height=cam.height, sh_degree=tc.sh_degree, backgrounds=bg.unsqueeze(0))
        except torch.cuda.OutOfMemoryError:
            raise TrainError(f"CUDA OOM with {model.num_gaussians} Gaussians. Reduce image size.")
        rendered = renders[..., :3]
        l1_loss = F.l1_loss(rendered, gt_image)
        ssim_val = _ssim(rendered, gt_image)
        loss = (1.0 - tc.loss_lambda_dssim) * l1_loss + tc.loss_lambda_dssim * (1.0 - ssim_val)
        if torch.isnan(loss):
            save_checkpoint(model, optimizer, iteration, ckpt_dir / f"iter_{iteration}_nan.pt")
            raise TrainError(f"NaN loss at iteration {iteration}")
        loss.backward()
        if model.means.grad is not None:
            model.grad_accum[:model.num_gaussians] += model.means.grad.norm(dim=-1, keepdim=True).detach()
            model.grad_count[:model.num_gaussians] += 1
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
        if tc.densify_from_iter <= iteration < tc.densify_until_iter and iteration % tc.densify_interval == 0 and iteration > 0:
            stats = _densify_and_prune(model, optimizer, config, iteration)
            log.info(f"  Densify @{iteration}: +{stats['clone']}c +{stats['split']}s -{stats['prune']}p -> {model.num_gaussians}")
        if iteration % tc.log_interval == 0:
            psnr = 10.0 * math.log10(1.0 / max(l1_loss.item(), 1e-10))
            metrics_log.append({"iteration": iteration, "loss": round(loss.item(), 6), "psnr": round(psnr, 2), "ssim": round(ssim_val.item(), 4), "num_gaussians": model.num_gaussians})
            if iteration % (tc.log_interval * 10) == 0:
                log.info(f"  iter {iteration}/{tc.iterations}: loss={loss.item():.4f} psnr={psnr:.1f} gs={model.num_gaussians} {time.time()-t_start:.0f}s")
        if iteration > 0 and iteration % tc.checkpoint_interval == 0:
            save_checkpoint(model, optimizer, iteration, ckpt_dir / f"iter_{iteration}.pt")
            log.info(f"  Checkpoint: iter_{iteration}.pt")
    final_path = train_dir / "final_model.pt"
    save_checkpoint(model, optimizer, tc.iterations, final_path)
    with open(train_dir / "metrics.json", "w") as f: json.dump(metrics_log, f, indent=2)
    elapsed = time.time() - t_start
    last = metrics_log[-1] if metrics_log else {}
    report = {"iterations": tc.iterations, "final_gaussians": model.num_gaussians, "final_loss": last.get("loss",0), "final_psnr": last.get("psnr",0), "final_ssim": last.get("ssim",0), "training_time_seconds": round(elapsed,1)}
    with open(run_dir / STAGE_MARKER, "w") as f: json.dump(report, f, indent=2)
    log.info(f"[bold green]Stage 3 (Train): complete — {model.num_gaussians} Gs, PSNR={report['final_psnr']}, {elapsed:.0f}s[/]")
    return report

In [ ]:
%%writefile /content/pipeline/splatagent_pipeline/stages/export.py
"""Stage 4 — Export .ply + manifest."""
from __future__ import annotations
import json, platform, shutil, subprocess, sys
from pathlib import Path
from ..config import PipelineConfig, config_to_dict
from ..logging_setup import get_logger
from ..manifest import BoundsInfo, Manifest, OutputInfo, SfMInfo, SourceInfo, ToolchainInfo, TrainingInfo
from ..utils.ply_io import write_ply

class ExportError(Exception): pass

def is_complete(run_dir):
    return (run_dir / "output" / "scene.ply").exists() and (run_dir / "output" / "manifest.json").exists()

def _get_tool_version(cmd):
    try: return subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.split("\n")[0].strip()
    except: return "unknown"

def run_export(config, run_dir, scene_id):
    import numpy as np
    import torch
    from .. import __version__
    log = get_logger()
    if config.run.resume and is_complete(run_dir):
        log.info("[bold green]Stage 4 (Export): skipping[/]")
        with open(run_dir / "output" / "manifest.json") as f: return json.load(f)
    log.info("[bold cyan]Stage 4 (Export): starting[/]")
    output_dir = run_dir / "output"
    output_dir.mkdir(parents=True, exist_ok=True)
    model_path = run_dir / "train" / "final_model.pt"
    if not model_path.exists(): raise ExportError(f"No trained model at {model_path}")
    data = torch.load(model_path, map_location="cpu", weights_only=True)
    means = data["means"].numpy().astype(np.float32)
    scales = data["scales"].numpy().astype(np.float32)
    rotations = data["rotations"].numpy().astype(np.float32)
    opacities = data["opacities"].numpy().astype(np.float32)
    sh_dc = data["sh_dc"].numpy().astype(np.float32).squeeze(1)
    sh_rest = data["sh_rest"].numpy().astype(np.float32)
    ply_path = output_dir / "scene.ply"
    n = write_ply(ply_path, means, scales, rotations, opacities, sh_dc, sh_rest, sh_degree=config.export.max_sh_degree)
    log.info(f"Wrote {n} Gaussians to {ply_path.name} ({ply_path.stat().st_size/1024/1024:.1f} MB)")
    bounds_min = means.min(axis=0).tolist()
    bounds_max = means.max(axis=0).tolist()
    def _load(p):
        if p.exists():
            with open(p) as f: return json.load(f)
        return {}
    ir, sr, tr = _load(run_dir/"ingest_report.json"), _load(run_dir/"sfm_report.json"), _load(run_dir/"train_report.json")
    cuda_ver = torch.version.cuda or "" if torch.cuda.is_available() else ""
    gsplat_ver = "unknown"
    try:
        import gsplat; gsplat_ver = getattr(gsplat, "__version__", "installed")
    except: pass
    manifest = Manifest(
        scene_id=scene_id,
        source=SourceInfo(type=ir.get("source_type",""), frame_count=ir.get("final_frame_count",0), image_dimensions=tuple(ir.get("image_dimensions",[0,0]))),
        sfm=SfMInfo(registered_images=sr.get("registered_images",0), sparse_points=sr.get("sparse_points",0), mean_reprojection_error=sr.get("mean_reprojection_error",0)),
        training=TrainingInfo(iterations=tr.get("iterations",0), final_gaussians=tr.get("final_gaussians",0), final_psnr=tr.get("final_psnr",0), final_ssim=tr.get("final_ssim",0), training_time_seconds=tr.get("training_time_seconds",0)),
        bounds=BoundsInfo(min=bounds_min, max=bounds_max),
        output=OutputInfo(ply_file="scene.ply", ply_size_bytes=ply_path.stat().st_size, coordinate_convention="colmap_y_down"),
        toolchain=ToolchainInfo(python_version=platform.python_version(), gsplat_version=gsplat_ver, colmap_version=_get_tool_version(["colmap","-h"]), torch_version=torch.__version__, cuda_version=cuda_ver, pipeline_version=__version__, ffmpeg_version=_get_tool_version(["ffmpeg","-version"])),
        config_snapshot=config_to_dict(config),
    )
    manifest.write(output_dir / "manifest.json")
    log.info(f"[bold green]Stage 4 (Export): complete[/]")
    with open(output_dir / "manifest.json") as f: return json.load(f)

In [ ]:
# Add pipeline to Python path
import sys
sys.path.insert(0, '/content/pipeline')

# Quick import test
from splatagent_pipeline.config import PipelineConfig, config_to_dict
from splatagent_pipeline.manifest import Manifest
print('Pipeline modules loaded OK')

---
## 2. Upload Data (Flowers Scene)

1. Click the **folder icon** in the left sidebar
2. Click the **upload icon** (up arrow)
3. Select `flowers_colab.zip` from your Desktop (192 MB)
4. Wait for the upload to complete, then run the cell below

In [ ]:
# === UNZIP + SET UP RUN DIRECTORY ===

!unzip -q /flowers_colab.zip -d /content/flowers_data

import os, shutil, json
from pathlib import Path

scene_id = "flowers"
run_dir = Path(f"/content/runs/{scene_id}")
run_dir.mkdir(parents=True, exist_ok=True)

# Symlink images (training reads from run_dir/images/)
images_src = Path("/content/flowers_data/flowers/images_4")
images_dst = run_dir / "images"
if not images_dst.exists():
    os.symlink(images_src, images_dst)

# Copy COLMAP sparse model (training reads from run_dir/sparse/0/)
sparse_dst = run_dir / "sparse" / "0"
sparse_dst.mkdir(parents=True, exist_ok=True)
sparse_src = Path("/content/flowers_data/flowers/sparse/0")
for f in ["cameras.bin", "images.bin", "points3D.bin"]:
    shutil.copy2(sparse_src / f, sparse_dst / f)

# Write stage reports (so export has metadata)
n_images = len(list(images_src.glob("*.JPG")))
ingest_report = {
    "source_type": "images", "source_path": str(images_src),
    "final_frame_count": n_images, "image_dimensions": [1262, 832],
    "blur_rejected": 0, "dedup_rejected": 0,
}
with open(run_dir / "ingest_report.json", "w") as f:
    json.dump(ingest_report, f, indent=2)

sfm_report = {
    "total_images": n_images, "registered_images": n_images,
    "sparse_points": 50000, "mean_reprojection_error": 0.5,
}
with open(run_dir / "sfm_report.json", "w") as f:
    json.dump(sfm_report, f, indent=2)

# Training iterations (7000 = quick test, 30000 = quality)
ITERATIONS = 7000

print(f"Scene ID: {scene_id}")
print(f"Run dir:  {run_dir}")
print(f"Images:   {n_images} files")
print(f"COLMAP:   {list(sparse_dst.iterdir())}")
print(f"Iters:    {ITERATIONS}")

## 3. Run Pipeline (Stages 3 & 4 only — COLMAP already done)

In [ ]:
import sys, dataclasses
sys.path.insert(0, '/content/pipeline')
from pathlib import Path
from splatagent_pipeline.config import PipelineConfig, RunConfig, IngestConfig, TrainConfig
from splatagent_pipeline.logging_setup import setup_logging

config = PipelineConfig(
    run=RunConfig(scene_id=scene_id, output_dir="/content/runs", seed=42, resume=True),
    ingest=IngestConfig(source_type="images", source_path=str(run_dir / "images")),
    train=dataclasses.replace(TrainConfig(), iterations=ITERATIONS),
)

setup_logging(run_dir)
print(f"Config ready. Run dir: {run_dir}")

In [ ]:
# Stage 1 (Ingest): SKIPPED — using pre-existing images_4
# Stage 2 (SfM):    SKIPPED — using pre-computed COLMAP sparse model
print("Stages 1 & 2 skipped (pre-computed COLMAP data)")
print(f"Images: {len(list((run_dir / 'images').iterdir()))} files")
print(f"Sparse: {list((run_dir / 'sparse' / '0').iterdir())}")

In [ ]:
# Stage 3: Gaussian Splat Training
from splatagent_pipeline.stages.train import run_train
train_report = run_train(config, run_dir)
print(f"\nGaussians: {train_report['final_gaussians']}")
print(f"PSNR: {train_report['final_psnr']}, SSIM: {train_report['final_ssim']}")
print(f"Training time: {train_report['training_time_seconds']:.0f}s")

In [ ]:
# Stage 4: Export .ply + manifest
from splatagent_pipeline.stages.export import run_export
manifest = run_export(config, run_dir, scene_id)

ply_path = run_dir / "output" / "scene.ply"
print(f"\nPLY: {ply_path} ({ply_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Manifest: {run_dir / 'output' / 'manifest.json'}")

## 4. Download Results

In [ ]:
# Download the .ply file
from google.colab import files

ply_path = run_dir / "output" / "scene.ply"
manifest_path = run_dir / "output" / "manifest.json"

print("Downloading scene.ply...")
files.download(str(ply_path))

print("Downloading manifest.json...")
files.download(str(manifest_path))

In [ ]:
# Alternative: Copy to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest = Path('/content/drive/MyDrive/splatagent_output') / scene_id
# dest.mkdir(parents=True, exist_ok=True)
# shutil.copy2(ply_path, dest / 'scene.ply')
# shutil.copy2(manifest_path, dest / 'manifest.json')
# print(f'Saved to {dest}')

## 5. Inspect Training Metrics (Optional)

In [ ]:
import json
import matplotlib.pyplot as plt

metrics_path = run_dir / "train" / "metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    iters = [m["iteration"] for m in metrics]
    psnrs = [m["psnr"] for m in metrics]
    losses = [m["loss"] for m in metrics]
    gs_counts = [m["num_gaussians"] for m in metrics]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(iters, psnrs); axes[0].set_title('PSNR'); axes[0].set_xlabel('Iteration')
    axes[1].plot(iters, losses); axes[1].set_title('Loss'); axes[1].set_xlabel('Iteration')
    axes[2].plot(iters, gs_counts); axes[2].set_title('Gaussians'); axes[2].set_xlabel('Iteration')
    plt.tight_layout(); plt.show()
else:
    print('No metrics file found')

---

**Done!** Drag the downloaded `scene.ply` into the SplatAgent viewer to view your 3D Gaussian Splat.

**Tips:**
- More iterations (30000) = better quality but ~30 min on T4
- 7000 iterations is a good quick test (~5 min)
- If COLMAP fails, your images may lack overlap or texture
- If training OOMs, reduce `max_dimension` in the ingest config